In [43]:
# Importing NumPy and setting up the dataset

import numpy as np

data = [
     [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]

In [44]:
# Step 1 (Encoding the Dataset)

for sample in data:
  if sample[3] == 'Wine':
    sample[3] = 0
  elif sample[3] == 'Beer':
    sample[3] = 1
  elif sample[3] == 'Whiskey':
    sample[3] = 2

data = np.array(data)
X = data[:, [0, 1, 2]].copy()
y = data[:, 3].copy()

print("Data with encoded labels:")
print(data)
print()
print("Feature Matrix X:")
print(X)
print()
print("Label Vector y:")
print(y)

Data with encoded labels:
[[12.   1.5  1.   0. ]
 [ 5.   2.   0.   1. ]
 [40.   0.   1.   2. ]
 [13.5  1.2  1.   0. ]
 [ 4.5  1.8  0.   1. ]
 [38.   0.1  1.   2. ]
 [11.5  1.7  1.   0. ]
 [ 5.5  2.3  0.   1. ]]

Feature Matrix X:
[[12.   1.5  1. ]
 [ 5.   2.   0. ]
 [40.   0.   1. ]
 [13.5  1.2  1. ]
 [ 4.5  1.8  0. ]
 [38.   0.1  1. ]
 [11.5  1.7  1. ]
 [ 5.5  2.3  0. ]]

Label Vector y:
[0. 1. 2. 0. 1. 2. 0. 1.]


In [45]:
# Step 2 (Implementing Gini Impurity)

def gini(labels):
  list_labels = labels.tolist()
  set_of_labels = set(labels)
  result = 1
  for label in set_of_labels:
    result -= (list_labels.count(label)/len(labels))**2
  return result

In [46]:
# Step 3 (Implementing the Best Split Finder)

def split(X, y):
  best_feature = None
  best_threshold = None
  min_wtd_gini = 2

  # Looping over all possible features to find the best one
  for i in range(X.shape[1]):
    feature = X[:, i]
    # Looping over all possible thresholds to find the best one
    for j in range(X.shape[0]-1):
      # Finding the candidate threshold (midpoint of the feature values for two consecutive samples)
      threshold = (feature[j] + feature[j+1])/2
      left = []
      right = []
      for k in range(X.shape[0]):
        if feature[k] < threshold:
          left.append(y[k])
        else:
          right.append(y[k])
      left = np.array(left)
      right = np.array(right)

      if (((len(left))*(gini(left)) + (len(right))*(gini(right)))/(len(left) + len(right))) < min_wtd_gini:
        min_wtd_gini = ((len(left))*(gini(left)) + (len(right))*(gini(right)))/(len(left) + len(right))
        best_feature = i
        best_threshold = threshold

  return [best_feature, best_threshold]

In [47]:
# Step 4 (Implementing Recursive Tree Building)

class Node:
  # Initializing
  def __init__(self, feature_index, threshold, X, y):
    self.feature_index = feature_index
    self.threshold = threshold
    self.left = None
    self.right = None
    self.value = None
    self.X = X
    self.y = y

  def create_tree(self):
    # Check if it is a leaf node
    if gini(self.y) == 0:
      self.left = None
      self.right = None
      self.value = self.y[0].item()
    else:
      # If it is not a leaf node, build the left and right child nodes
      X_left = self.X[self.X[:, self.feature_index] < self.threshold].copy()
      X_right = self.X[self.X[:, self.feature_index] >= self.threshold].copy()
      y_left = []
      y_right = []
      for i in range(len(self.y)):
        if self.X[i][self.feature_index] < self.threshold:
          y_left.append(self.y[i])
        else:
          y_right.append(self.y[i])
      y_left = np.array(y_left)
      y_right = np.array(y_right)

      # Using the split function to find the best split for the upcoming left and right children
      self.left = Node(split(X_left, y_left)[0], split(X_left, y_left)[1], X_left, y_left)
      self.right = Node(split(X_right, y_right)[0], split(X_right, y_right)[1], X_right, y_right)

      # Recursively building the tree
      self.left.create_tree()
      self.right.create_tree()

tree = Node(split(X, y)[0], split(X, y)[1], X, y)
tree.create_tree()

In [48]:
# Step 5 (Implementing the Prediction)

def predict_one(tree, x):
  if tree.value != None:
    return tree.value
  elif x[tree.feature_index] < tree.threshold:
    # Recursively traversing the tree
    return predict_one(tree.left, x)
  else:
    # Recursively traversing the tree
    return predict_one(tree.right, x)

def predict(tree, test_data):
  predictions = []
  for x in test_data:
    predictions.append(predict_one(tree, x))
  return predictions

In [49]:
# Step 6 (Evaluation)

test_data = np.array([
    [6.0, 2.1, 0],
    [39.0, 0.05, 1],
    [13.0, 1.3, 1]
])

predictions = predict(tree, test_data)
print("Predicted labels for the test data:")
print(predictions)
print()

if predictions == [1, 2, 0]:
  print("The Tree predicted the labels accurately")
else:
  print("The Tree did not predict the labels accurately")

Predicted labels for the test data:
[1.0, 2.0, 0.0]

The Tree predicted the labels accurately
